# Phase 7 — Deterministic Security Rule Engine: Calibration & ML Complementarity

**Status:** ✅ Completed (pipeline v1.0)  
**Authoritative Scripts:** `ml_service/src/rules/rule_engine.py` & `ml_service/src/rules/evaluate_rules.py`  
**Rule Catalog:** `artifacts/rules/rule_definitions_v1.json`  
**Evaluation Report:** `artifacts/rules/rule_evaluation_v1.md`  
**Metrics Artifact:** `artifacts/rules/rule_metrics_v1.json`  

---

## Executive Summary & Research Scope
Phase 7 establishes the deterministic security rule engine operating independently of ML predictions per AGENTS.md.
Key methodological controls enforced:
- **Raw Unscaled Telemetry:** Rules evaluate raw continuous metrics and `Destination Port` from `artifacts/cleaned/*.csv` (not z-score scaled features).
- **Row Alignment Assertion:** Verified 100% element-wise match (`np.array_equal`) between extracted test labels and `y_test_category_v1.parquet` across all **504,160 flows**.
- **Train-Only Calibration:** Thresholds were evaluated on the training split, and a strict **50% precision floor** was enforced to prevent false alert flooding into Phase 8.
- **Measured Latency:** Empirically clocked at **0.25 μs/sample**.
- **ML Complementarity Matrix:** Mapped exact overlap against the primary XGBoost classifier.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths
from ml_service.src.rules.rule_engine import SecurityRuleEngine

# Load rule definitions and evaluation metrics
with open(Paths.ARTIFACTS_DIR / 'rules' / 'rule_definitions_v1.json', 'r', encoding='utf-8') as f:
    definitions_data = json.load(f)
with open(Paths.ARTIFACTS_DIR / 'rules' / 'rule_metrics_v1.json', 'r', encoding='utf-8') as f:
    metrics_data = json.load(f)

print('Loaded Phase 7 Rule Engine Artifacts Successfully.')
print(f"Test Flows Evaluated: {metrics_data['test_flows_count']:,}")
print(f"Empirical Evaluation Latency: {metrics_data['empirical_latency_us_per_flow']:.2f} us/flow")
print(f"Ensemble Precision: {metrics_data['ensemble_metrics']['precision']:.2%}")

## 1. Calibrated Rule Set & Training Precision Audit
Rules were tested against a **50% precision floor** on training data. Rules failing the floor were demoted to `INFORMATIONAL` status to safeguard Phase 8 correlation from alert flooding:

In [ ]:
rules_list = definitions_data['rules']
rows = []
for r in rules_list:
    m = metrics_data['per_rule_metrics'][r['rule_id']]
    train_p = f"{r['precision_train']:.2%}" if r.get('precision_train') is not None else 'N/A'
    rows.append({
        'Rule ID': r['rule_id'],
        'Threat': r['target_threat'],
        'Severity': r['severity'],
        'Status': 'ACTIVE' if r['is_active'] else 'INFORMATIONAL',
        'Train Precision': train_p,
        'Test Precision': f"{m['precision']:.2%}",
        'Test Recall (TPR)': f"{m['recall_tpr']:.4f}",
        'Test FPR': f"{m['fpr']:.4%}",
        'Test TP': f"{m['true_positives']:,}",
        'Test FP': f"{m['false_positives']:,}"
    })

df_rules = pd.DataFrame(rows)
display(df_rules)

## 2. ML vs. Rule Complementarity Matrix
We compare detections between the primary XGBoost classifier (Phase 5) and the calibrated deterministic Rule Engine across the full 504,160-row test split:
- **Category A (Both Flagged):** High-confidence synergy on credential guessing handshakes.
- **Category B (ML Only):** Complex volumetric, inter-arrival, and geometric attacks captured by ML non-linear boundaries that simple static thresholds cannot express.
- **Category C (Rules Only):** Administrative port traffic where ML predicted Benign.
- **Category D (Neither Flagged):** Clean benign network flows passed by both systems.

In [ ]:
comp = metrics_data['complementarity_matrix']
n_total = metrics_data['test_flows_count']

comp_df = pd.DataFrame([
    {'Category': 'Category A (ML & Rules Flagged)', 'Flow Count': comp['category_A_both_flag'], 'Percentage': f"{comp['category_A_both_flag']/n_total:.2%}", 'Role': 'True Threat Synergy'},
    {'Category': 'Category B (ML Only Flagged)', 'Flow Count': comp['category_B_ml_only'], 'Percentage': f"{comp['category_B_ml_only']/n_total:.2%}", 'Role': 'ML High-Dimensional Generalization'},
    {'Category': 'Category C (Rules Only Flagged)', 'Flow Count': comp['category_C_rules_only'], 'Percentage': f"{comp['category_C_rules_only']/n_total:.2%}", 'Role': 'Deterministic Safety Net / Edge Rescue'},
    {'Category': 'Category D (Neither Flagged)', 'Flow Count': comp['category_D_neither_flag'], 'Percentage': f"{comp['category_D_neither_flag']/n_total:.2%}", 'Role': 'Clean Benign Traffic'}
])

display(comp_df)

# Visual Breakdown
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
labels = ['Cat A: Both', 'Cat B: ML Only', 'Cat C: Rules Only']
counts = [comp['category_A_both_flag'], comp['category_B_ml_only'], comp['category_C_rules_only']]
colors = ['#2ca02c', '#1f77b4', '#ff7f0e']

bars = ax.bar(labels, counts, color=colors, edgecolor='black', alpha=0.85)
ax.set_ylabel('Number of Flows', fontsize=10, fontweight='bold')
ax.set_title('Threat Detection Breakdown: XGBoost ML Classifier vs. Rule Engine', fontsize=11, fontweight='bold')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1000, f'{height:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Local Flow Verification: `evaluate_flow` Utility
We demonstrate evaluating individual flow dictionaries with `SecurityRuleEngine.evaluate_flow()` returning structured diagnostic violation tags for SOC analysts:

In [ ]:
engine = SecurityRuleEngine.load_definitions(Paths.ARTIFACTS_DIR / 'rules' / 'rule_definitions_v1.json')

# Example 1: Brute force handshake on port 22
flow_ssh = {
    'Destination Port': 22,
    'min_seg_size_forward': 32,
    'Total Backward Packets': 12,
    'Flow Duration': 500000,
    'SYN Flag Count': 1,
    'ACK Flag Count': 1,
}
result_ssh = engine.evaluate_flow(flow_ssh)
print('SSH Brute Force Flow Evaluation:')
print(json.dumps(result_ssh, indent=2))

# Example 2: Normal web traffic on port 443
flow_benign = {
    'Destination Port': 443,
    'min_seg_size_forward': 20,
    'Total Backward Packets': 4,
    'Flow Duration': 12000,
    'SYN Flag Count': 0,
    'ACK Flag Count': 1,
}
result_benign = engine.evaluate_flow(flow_benign)
print('\nBenign Web Flow Evaluation:')
print(json.dumps(result_benign, indent=2))